In [2]:

import pandas as pd
import numpy as np

RAW_PATH = "iot_sensor_raw_data_extended.csv"
CLEANED_PATH = "iot_sensor_cleaned.csv"

# ---------- 1. โหลด Raw Data (เก็บต้นฉบับไว้เสมอ ไม่แก้ไขไฟล์นี้) ----------
raw_df = pd.read_csv(RAW_PATH)
df = raw_df.copy()
n_before = len(df)

issue_log = []

# ---------- 2. ตรวจสอบปัญหา "ก่อน" ทำความสะอาด (สำหรับรายงาน) ----------
dup_mask_before = df.duplicated(subset=["timestamp", "device_id"], keep=False)
temp_outlier_before = df["temp_c"].notna() & ((df["temp_c"] < 0) | (df["temp_c"] > 50))
batt_outlier_before = df["battery"].notna() & ((df["battery"] < 0) | (df["battery"] > 100))
missing_device_before = df["device_id"].isna()
missing_temp_before = df["temp_c"].isna()
missing_batt_before = df["battery"].isna()
valid_motion = {"true", "false"}
motion_error_before = ~df["motion"].isin(valid_motion) & df["motion"].notna()
missing_motion_before = df["motion"].isna()

summary_before = {
    "Duplicate rows (รวมทั้งชุด)": int(dup_mask_before.sum()),
    "Temp outlier (นอกช่วง 0-50)": int(temp_outlier_before.sum()),
    "Battery outlier (นอกช่วง 0-100)": int(batt_outlier_before.sum()),
    "Missing device_id": int(missing_device_before.sum()),
    "Missing temp_c": int(missing_temp_before.sum()),
    "Missing battery": int(missing_batt_before.sum()),
    "Motion ค่าไม่ถูกต้อง (error)": int(motion_error_before.sum()),
    "Motion ว่าง (missing)": int(missing_motion_before.sum()),
}

# ---------- 3. ลบข้อมูลซ้ำ (Duplicate) ----------
before_dedup = len(df)
df = df.drop_duplicates(subset=["timestamp", "device_id"], keep="first").reset_index(drop=True)
n_duplicates_removed = before_dedup - len(df)

# ---------- 4. แปลงค่าผิดปกติ (Error/Outlier) เป็น NULL ตาม Data Quality Rules ----------
# R1: temp_c ต้องอยู่ 0-50
temp_bad = (df["temp_c"] < 0) | (df["temp_c"] > 50)
df.loc[temp_bad, "temp_c"] = np.nan

# R4: battery ต้องอยู่ 0-100
batt_bad = (df["battery"] < 0) | (df["battery"] > 100)
df.loc[batt_bad, "battery"] = np.nan

# R5 (เพิ่มเติม): motion ต้องเป็น true/false เท่านั้น ค่าอื่นถือเป็น error -> NULL ก่อน
df["motion"] = df["motion"].str.strip().str.lower()
motion_bad = ~df["motion"].isin(valid_motion) & df["motion"].notna()
df.loc[motion_bad, "motion"] = np.nan

# ---------- 5. จัดการ Missing Value ----------
# แปลง timestamp เป็น datetime และเรียงลำดับตาม device เพื่อ interpolate ตามลำดับเวลา
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["device_id", "timestamp"]).reset_index(drop=True)

# 5.1 device_id ว่าง (R2) -> ไม่มีทางเติมค่าคืนได้อย่างน่าเชื่อถือ (ไม่รู้ว่าเป็นเซนเซอร์ตัวไหน)

invalid_device_mask = df["device_id"].isna()

# 5.2 temp_c ว่าง (รวมค่าที่ถูกแปลงจาก outlier ในขั้นตอนที่ 4)
df["temp_c"] = df.groupby("device_id", group_keys=False)["temp_c"].apply(
    lambda s: s.interpolate(limit_direction="both")
)
# ถ้ายังว่างอยู่ (เช่น device มีค่าว่างทั้งหมด) ให้เติมด้วยค่ามัธยฐานรวม
df["temp_c"] = df["temp_c"].fillna(df["temp_c"].median())

# 5.3 battery ว่าง (รวมค่าที่ถูกแปลงจาก outlier)
#     แบตเตอรี่มักลดลงอย่างช้า ๆ -> interpolate เช่นกัน
df["battery"] = df.groupby("device_id", group_keys=False)["battery"].apply(
    lambda s: s.interpolate(limit_direction="both")
)
df["battery"] = df["battery"].fillna(df["battery"].median())

# 5.4 motion ว่าง (รวมค่าที่ถูกแปลงจาก error) -> เติมด้วยค่าที่พบบ่อยที่สุด (mode) ของอุปกรณ์ตัวนั้น
def fill_motion(s):
    mode = s.mode(dropna=True)
    fill_val = mode.iloc[0] if not mode.empty else "false"
    return s.fillna(fill_val)

df["motion"] = df.groupby("device_id", group_keys=False)["motion"].apply(fill_motion)

# ---------- 6. เพิ่มคอลัมน์ data_quality_status ----------
# "invalid_missing_device_id" = แถวที่ไม่มี device_id (ไม่สามารถระบุที่มาของข้อมูลได้)
# "cleaned_ok" = แถวที่ผ่านการทำความสะอาด (แก้ error/outlier/missing แล้ว) และพร้อมใช้งาน
status = np.where(invalid_device_mask, "invalid_missing_device_id", "cleaned_ok")
df["data_quality_status"] = status

n_after = len(df)
n_invalid_device = int(invalid_device_mask.sum())

# ชุดข้อมูลสะอาดสุดท้าย: ตัดแถวที่ device_id ว่างออก (เก็บ log ไว้ในรายงาน แต่ไม่รวมใน dataset ที่ใช้งานต่อ)
clean_final = df[~invalid_device_mask].copy()

# ---------- 7. บันทึกไฟล์ผลลัพธ์ ----------
df.to_csv(CLEANED_PATH.replace(".csv", "_with_flagged_rows.csv"), index=False)
clean_final.to_csv(CLEANED_PATH, index=False)

# ---------- 8. สรุปผลก่อน-หลัง ----------
print("=== สรุปจำนวนแถวก่อน/หลังทำความสะอาด ===")
print(f"จำนวนแถวก่อนทำความสะอาด (Raw): {n_before}")
print(f"จำนวนแถวที่เป็น Duplicate ที่ถูกลบออก: {n_duplicates_removed}")
print(f"จำนวนแถวหลังลบ Duplicate: {n_after}")
print(f"จำนวนแถวที่ถูกตัดออกเพราะ device_id ว่าง (invalid): {n_invalid_device}")
print(f"จำนวนแถวในชุดข้อมูลสะอาดสุดท้าย: {len(clean_final)}")
print()
print("=== สรุปปัญหาที่พบก่อนทำความสะอาด ===")
for k, v in summary_before.items():
    print(f"{k}: {v}")

=== สรุปจำนวนแถวก่อน/หลังทำความสะอาด ===
จำนวนแถวก่อนทำความสะอาด (Raw): 480
จำนวนแถวที่เป็น Duplicate ที่ถูกลบออก: 25
จำนวนแถวหลังลบ Duplicate: 455
จำนวนแถวที่ถูกตัดออกเพราะ device_id ว่าง (invalid): 2
จำนวนแถวในชุดข้อมูลสะอาดสุดท้าย: 453

=== สรุปปัญหาที่พบก่อนทำความสะอาด ===
Duplicate rows (รวมทั้งชุด): 50
Temp outlier (นอกช่วง 0-50): 10
Battery outlier (นอกช่วง 0-100): 5
Missing device_id: 2
Missing temp_c: 7
Missing battery: 4
Motion ค่าไม่ถูกต้อง (error): 5
Motion ว่าง (missing): 4
